In [7]:
!pip install linearmodels

  Using cached linearmodels-7.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (10 kB)
  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
  Using cached pyhdfe-0.2.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached formulaic-1.2.1-py3-none-any.whl.metadata (7.0 kB)
  Using cached interface_meta-1.3.0-py3-none-any.whl.metadata (6.7 kB)
Using cached linearmodels-7.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (1.5 MB)
Using cached formulaic-1.2.1-py3-none-any.whl (117 kB)
Using cached mypy_extensions-1.1.0-py3-none-any.whl (5.0 kB)
Using cached pyhdfe-0.2.0-py3-none-any.whl (19 kB)
Using cached interface_meta-1.3.0-py3-none-any.whl (14 kB)

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
db = pd.read_csv('./CleanData.csv')
Countries =pd.Series((pd.read_csv('./CountryCodes.csv').squeeze()))

# importing all the policy data
carbon_pricing_df = pd.read_csv("./data/IEA_carbon_pricing.csv")
coal_phasing_df = pd.read_csv("./data/IEA_coal_phase-out.csv")
emissions_standards_df = pd.read_csv("./data/IEA_emissions_standards.csv")
low_carbon_spending_df = pd.read_csv("./data/IEA_low_carbon_spending.csv")
policies_df = pd.concat([carbon_pricing_df, coal_phasing_df, emissions_standards_df, low_carbon_spending_df])
policies_df.head(1)


,title,description,status,year,jurisdiction,source,datePromulgated,yearEnded,learnMore,learnMoreLanguage,dateModified,countries,states,technologies,tags,policyType
0,Regional Greenhouse Gas Initiative (RGGI),<div>The Regional Greenhouse Gas Initiative (R...,In force,2008,State/Provincial,JOIN IEA/IRENA Policy and Measures Database,NaN,NaN,http://www.rggi.org/home,NaN,1711642099496,"[{""iso3"":""USA"",""name"":""United States""}]","[{""country"":""US"",""code"":""VT"",""name"":""Vermont""}]",[],[],"[{""topic"":""Power"",""family"":""Market pull"",""name..."


In [9]:
def PolicyCalc():
    outputList = []

    for i in Countries:
        for Year in range(1992, 2024):
            x = policies_df[policies_df['countries'].str.contains(i, na=False)]
            x = x[x['year'] <= Year]
            Policies = len(x)
            x = x[x['yearEnded'] < Year]
            # Reworked to catch the Ended policies with Null YearEnded
            y = policies_df[policies_df['status'] == 'Ended']
            y = policies_df[policies_df['yearEnded'].isnull()]
            y = y[y['year'] <= Year]
            
            Policies = Policies - len(x) 
            new_row = {'Code': i, 'Year': Year,'Policies': Policies}
            outputList.append(new_row)
    Z =  pd.DataFrame(outputList)
    return Z    

Z = PolicyCalc()

In [10]:
merged_df = Z.merge(db, on = ['Code', 'Year'])
merged_df.head()

,Code,Year,Policies,Entity,Share of energy from low-carbon sources,GDP per capita,World region according to OWID
0,DZA,1992,1,Algeria,0.172640,11241.415,Africa
1,DZA,1993,1,Algeria,0.329121,10743.706,Africa
2,DZA,1994,1,Algeria,0.152741,10414.035,Africa
3,DZA,1995,1,Algeria,0.171221,10588.443,Africa
4,DZA,1996,1,Algeria,0.121007,10808.879,Africa


In [5]:
#model
import statsmodels.api as sm
merged_df.rename(columns={'Share of energy from low-carbon sources': 'Share'}, inplace=True)
formula_string = "Share ~ Policies"

model = sm.formula.ols(formula = formula_string, data = merged_df)
model_fitted = model.fit()

print(model_fitted.summary())

                            OLS Regression Results                            
Dep. Variable:                  Share   R-squared:                       0.015
Model:                            OLS   Adj. R-squared:                  0.015
Method:                 Least Squares   F-statistic:                     38.23
Date:                Thu, 09 Apr 2026   Prob (F-statistic):           7.34e-10
Time:                        17:50:43   Log-Likelihood:                -10396.
No. Observations:                2464   AIC:                         2.080e+04
Df Residuals:                    2462   BIC:                         2.081e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     13.5367      0.456     29.710      0.0

In [11]:
from linearmodels.panel import PanelOLS
#imported from the package installed in cell 1
df1 = merged_df.set_index(['Code', 'Year'])

#Model with Entity and Time effects
#EntityEffects controls for fixed country traits (Norway has an easier time making green choices etc)
#TimeEffects controls for global year-by-year trends( The world is greener in 2020 vs 1990)
model = PanelOLS.from_formula('Share ~ Policies + EntityEffects + TimeEffects', data=df1)

#Model using Clustered Standard Errors by Country
results = model.fit(cov_type='clustered', cluster_entity=True)

print(results)

                          PanelOLS Estimation Summary                           
Dep. Variable:                  Share   R-squared:                        0.0366
Estimator:                   PanelOLS   R-squared (Between):              0.2616
No. Observations:                2464   R-squared (Within):              -40.665
Date:                Thu, Apr 09 2026   R-squared (Overall):             -1.0382
Time:                        12:38:49   Log-likelihood                   -6556.4
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      89.586
Entities:                          77   P-value                           0.0000
Avg Obs:                       32.000   Distribution:                  F(1,2355)
Min Obs:                       32.000                                           
Max Obs:                       32.000   F-statistic (robust):             5.7153
                            

In [12]:
#2 year time lag , I also tested 1 year 
df2 = merged_df
df2['Policies_Lag2'] = df2.groupby('Code')['Policies'].shift(2)


#dropna() because the first year (1992) will now have a NaN lag
model_lag = PanelOLS.from_formula(
    'Share ~ Policies_Lag2 + EntityEffects + TimeEffects', 
    data=df2.dropna(subset=['Policies_Lag2']).set_index(['Code', 'Year'])
)

results_lag = model_lag.fit(cov_type='clustered', cluster_entity=True)
print(results_lag)

                          PanelOLS Estimation Summary                           
Dep. Variable:                  Share   R-squared:                        0.0537
Estimator:                   PanelOLS   R-squared (Between):             -0.4330
No. Observations:                2310   R-squared (Within):              -65.333
Date:                Thu, Apr 09 2026   R-squared (Overall):             -2.4177
Time:                        12:38:49   Log-likelihood                   -6083.7
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      124.92
Entities:                          77   P-value                           0.0000
Avg Obs:                       30.000   Distribution:                  F(1,2203)
Min Obs:                       30.000                                           
Max Obs:                       30.000   F-statistic (robust):             10.478
                            

In [13]:
# Log of GDP pc is standard practise to normalise difference between large and small values. 
df2['Log_GDP_pc'] = np.log(df2['GDP per capita'])

#The data
df_final = df2.dropna(subset=['Policies_Lag2', 'Log_GDP_pc', 'Share']).set_index(['Code', 'Year'])


model_gdp = PanelOLS.from_formula(
    'Share ~ Policies_Lag2 + Log_GDP_pc + EntityEffects + TimeEffects', 
    data=df_final
)

results_gdp = model_gdp.fit(cov_type='clustered', cluster_entity=True)
print(results_gdp)

                          PanelOLS Estimation Summary                           
Dep. Variable:                  Share   R-squared:                        0.0722
Estimator:                   PanelOLS   R-squared (Between):              0.1533
No. Observations:                2310   R-squared (Within):              -51.263
Date:                Thu, Apr 09 2026   R-squared (Overall):             -1.4191
Time:                        12:38:49   Log-likelihood                   -6060.9
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      85.627
Entities:                          77   P-value                           0.0000
Avg Obs:                       30.000   Distribution:                  F(2,2202)
Min Obs:                       30.000                                           
Max Obs:                       30.000   F-statistic (robust):             6.6714
                            